In [0]:
%pip install apache-sedona==1.7.1

In [0]:
%restart_python

In [0]:
%pip install geopandas

In [0]:
from pyspark.sql.functions import expr, first, broadcast, col, when, coalesce, lower, trim, lit
from sedona.spark import SedonaContext

# 셔플 파티션 설정 및 Sedona 초기화
spark.conf.set("spark.sql.shuffle.partitions", "200")
sedona = SedonaContext.create(spark)
 
# Phase 1. 데이터 로드 및 공간 조인

# 반려견 산책 가능 도로 등급만 필터링
safe_highways = [
    "footway", "path", "pedestrian", "residential", 
    "living_street", "track", "steps"
]

# WKT → 공간 객체 변환
edges_clean = spark.table("bronze.osm_edges") \
    .filter(col("highway").isin(safe_highways)) \
    .withColumn("geom_e", expr("ST_GeomFromWKT(geometry)")) \
    .select("id", "length", col("u").alias("start_node"), col("v").alias("end_node"), 
            "surface", "smoothness", "highway", "geometry", "geom_e")

# 공간 조인 전 캐시 고정
edges_clean.cache()
edges_clean.count()
print("OSM 도로 데이터 로드 완료!")


# V-World 토양 5종 레이어 로드 및 공간 객체 변환
soil_type_clean  = spark.table("bronze.vworld_soil_type").select("DEEPSOIL", expr("ST_GeomFromWKT(geometry)").alias("geom_s"))
slope_clean      = spark.table("bronze.vworld_slope").select("SOILSLOPE", expr("ST_GeomFromWKT(geometry)").alias("geom_sl"))
gravel_clean     = spark.table("bronze.vworld_gravel").select("SUR_STON", expr("ST_GeomFromWKT(geometry)").alias("geom_g"))
soil_depth_clean = spark.table("bronze.vworld_soil_depth").select("VLDSOILDEP", expr("ST_GeomFromWKT(geometry)").alias("geom_sd"))
drainage_clean   = spark.table("bronze.vworld_drainage").select("SOILDRA", expr("ST_GeomFromWKT(geometry)").alias("geom_d"))


# 공간 조인 (ST_Distance < 0.00005)
# V-World 레이어는 행 수가 적어 Broadcast Join으로 셔플 오버헤드 제거
edges_joined = edges_clean \
    .join(broadcast(soil_type_clean),  expr("ST_Distance(geom_e, geom_s) < 0.00005"), "left") \
    .join(broadcast(slope_clean),      expr("ST_Distance(geom_e, geom_sl) < 0.00005"), "left") \
    .join(broadcast(gravel_clean),     expr("ST_Distance(geom_e, geom_g) < 0.00005"), "left") \
    .join(broadcast(soil_depth_clean), expr("ST_Distance(geom_e, geom_sd) < 0.00005"), "left") \
    .join(broadcast(drainage_clean),   expr("ST_Distance(geom_e, geom_d) < 0.00005"), "left")

print("공간 조인 완료!")

In [0]:
from pyspark.sql.functions import expr, first, avg, col, when, trim, coalesce, lit, round, lower

# Phase 2. 경사도 수치 변환 및 결측 보완

# 집계 단계 — 파티션 수 축소 (200 → 64)
spark.conf.set("spark.sql.shuffle.partitions", "64")

# 경사도 범위 문자열 → 구간 중앙값(%)으로 수치 변환
slope_numeric = when(trim(col("SOILSLOPE")) == "0-2%", 1.0) \
                .when(trim(col("SOILSLOPE")) == "2-7%", 4.5) \
                .when(trim(col("SOILSLOPE")) == "7-15%", 11.0) \
                .when(trim(col("SOILSLOPE")) == "15-30%", 22.5) \
                .when(trim(col("SOILSLOPE")) == "30-60%", 45.0) \
                .when(trim(col("SOILSLOPE")) == "60-100%", 80.0) \
                .otherwise(None)

# geometry를 groupBy에서 제외 — 셔플 데이터 볼륨 최소화
silver_df = edges_joined.withColumn("slope_val", slope_numeric).groupBy(
    "id", "start_node", "end_node"
).agg(
    first("geom_e").alias("geom"),
    first("length").alias("length"),
    first("surface").alias("surface"),
    first("smoothness").alias("smoothness"),
    first("highway").alias("highway"),
    first("DEEPSOIL",    True).alias("soil_type"),
    first("SUR_STON",    True).alias("gravel_content"),
    first("VLDSOILDEP",  True).alias("soil_depth"),
    first("SOILDRA",     True).alias("drainage_class"),
    avg("slope_val").alias("avg_slope_raw")   # 경사도만 avg, 나머지는 first
)

# 잔여 NULL 처리 후 캐시 고정
silver_df = silver_df.withColumn("soil_type", coalesce(col("soil_type"), lit("Unknown")))
silver_df.cache()

# 미매칭 구간 결측 보완 — highway 유형별 기본값 부여
# steps는 수치 경사도 의미 없으므로 NULL 처리
silver_df = silver_df.withColumn(
    "avg_slope",
    when(col("highway") == "steps", lit(None))
    .when(col("avg_slope_raw").isNotNull(), col("avg_slope_raw"))
    .otherwise(
        when(col("highway") == "track", 7.0)
        .when(col("highway").isin("living_street", "path"), 3.5)
        .when(col("highway").isin("footway"), 2.5)
        .otherwise(1.5)
    )
).withColumn(
    # 평지(≤3%) / 완만(≤8%) / 급경사(>8%) / 계단 분류
    "slope_type",
    when(col("highway") == "steps", lit("계단"))
    .when(col("avg_slope").isNull(), lit("정보없음"))
    .when(col("avg_slope") <= 3.0, lit("평지"))
    .when(col("avg_slope") <= 8.0, lit("완만"))
    .otherwise(lit("급경사"))
)

# 소수점 정리 및 텍스트 정규화
silver_df = silver_df.withColumn("avg_slope", round(col("avg_slope"), 1)) \
                     .withColumn("surface", lower(trim(col("surface")))) \
                     .withColumn("soil_type", trim(col("soil_type"))) \
                     .drop("avg_slope_raw")

# 이후 Score 산출을 위해 캐시 고정
silver_df.cache()

print(f" 경사도 데이터 정제 및 보정 성공 (최종 행수: {silver_df.count()})")

# 1. 데이터베이스 생성 (이미 있다면 무시)
spark.sql("CREATE DATABASE IF NOT EXISTS silver")

# 2. 저장 전 파티션 최적화
# 최종 행 수가 22만 건 정도라면, 파일 갯수를 적절히 조절해서 저장하는 게 나중에 Gold에서 부를 때 더 빠름
silver_df_final = silver_df.withColumn("geometry", expr("ST_AsText(geom)")) \
                           .drop("geom") \
                           .repartition(10)

# 3. Delta 테이블로 저장
silver_df_final.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.walk_features")

print("Silver 레이어: silver.walk_features 저장 완료!")

In [0]:
from pyspark.sql.functions import monotonically_increasing_id, col, when, lit, lower, coalesce, expr

# 1. ID 부여
leisure_with_id = spark.table("bronze.osm_leisure") \
    .withColumn("id", monotonically_increasing_id()) 

# 2. 키워드 패턴
EXCLUDE_KEYWORDS = ["아파트", "잠실주공5단지", "아크로힐스", "로렌하우스", "대치에스케이뷰", "래미안대치"]
pattern = "|".join(EXCLUDE_KEYWORDS)

# 3. 오분류 데이터 필터링 및 공간 객체 변환
poi_silver = leisure_with_id.filter(
    col("leisure").isin("park", "dog_park") & 
    (~coalesce(col("name"), lit("")).rlike(pattern))
).select(
    "id",
    when(col("name").isNull(), lit("장소명 없음")).otherwise(col("name")).alias("name"),
    "leisure",
    "geometry"
).withColumn("geom_p", expr("ST_GeomFromWKT(geometry)"))

# 4. 저장
poi_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.park")

print("silver.park 테이블 생성 완료!")

In [0]:
walk_features = spark.table("silver.park")
walk_features.printSchema()